# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=****1434
OLLAMA_MODEL=****:e4b


In [2]:
from langchain.agents import create_agent

In [3]:
import os

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

**(The comedian steps up to a non-existent mic stand, adjusts an imaginary earpiece, and beams at the audience.)**

Alright, alright, settle down! You want a joke? You came to the right place. I'm a *full-stack* comedian. That means I can handle anything: I can do observational comedy on the merits of lukewarm coffee, I can do self-deprecating stand-up about my crippling inability to assemble IKEA furniture, and I can drop a niche nerd joke so specific, you’ll think I’m talking about database architecture.

The risk, however, is that I might switch stacks mid-set. You've been warned.

*(Pauses for dramatic effect, leans in conspiratorially.)*

So, here's one. It’s observational. It’s relatable. It touches on the universal modern struggle that we all pretend we don't notice: **smartphones.**

Why did the iPhone break up with the Android?

...Because it said, "You're always so worried about compatibility! We just don't *sync* anymore."

*(The comedian waits, stares into the void, and the

## values
You have seen this streaming mode in our examples so far. 

In [5]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

**(Prepares self, clears throat dramatically, maybe taps the microphone twice—a little too loudly.)**

Alright, alright. You asked for a Dad joke. You didn't ask for a complex, multi-layered piece of observational comedy that requires an understanding of postmodern irony and the socioeconomic implications of sourdough starter, no. You asked for the deep, comforting, low-resource memory of pure, unadulterated cringe.

Consider this my **full-stack deployment** of dad humor. I’ve handled the front-end presentation (the setup), the back-end logic (the pun construction), and the database integrity (the universal groan).

Here it is. Brace yourself.

***

Why don't scientists trust atoms?

**(Pauses, looking extremely serious, as if revealing the secrets of the universe.)**

Because they make up *everything!*



## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [6]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

*(The comedian paces the stage, adjusts an invisible lapel, and beams at the audience.)*

"A family-friendly poem? Okay. So, I'm a full-stack comedian. I write code, I roast people, I've seen deep, dark comedy material... but you want *family friendly*? You want something wholesome? You want me to rhyme about kittens and bedtime? Challenge accepted. My rhymes are tighter than my escape artist act, and my punchlines are safer than a petting zoo.

Here is my ode to the glorious, messy, wonderful chaos that is family life."

***

### The Ode to the Great Day Machine

The sun pops up, a cheerful, yellow sight,
And wakes us up with golden, beaming light.
(Though some of us still dream of sleeping through the day,
We gotta make the sunshine bloom and play!)

We throw our sweaters on and put on hiking shoes,
Forgetting all the napping and the cozy blues.
We chase the squirrels up the biggest oak tree high,
And practice leaping with a big, enthusiastic *fly!*

We laugh until our faces start to

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [7]:
import os

from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='5e3cff50-f557-4758-b2cf-f7eff4289b55')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='5e3cff50-f557-4758-b2cf-f7eff4289b55'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-04T06:40:15.047764Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12692433708, 'load_duration': 111971375, 'prompt_eval_count': 66, 'prompt_eval_duration': 200797417, 'eval_count': 308, 'eval_duration': 12299376625, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d5738-5332-7a12-a44f-9053ba2280c8-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'a6332086-ecb5-45bf-8fa9-4bdaa67384b5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 3

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: San Francisco')
('custom', 'Acquired data for city: San Francisco')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [9]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

Looking up data for city: SF
Acquired data for city: SF
